# Emerging Technologies Assignment 
- Nora Keavney
- G00415845

- Overall Explanation for Assignment --------
- Purpose in grand scheme 
- Link to References ...

## Problem 1: Generating Random Boolean Functions


### Problem Context

This problem creates example Boolean functions that will be used in later parts of the assignment.
The Deutsch–Jozsa algorithm assumes that a function is either constant or balanced, and its goal is to determine which type it is without knowing how the function works internally.

Problem 1 provides these functions in a classical form so they can be tested and analysed in both classical and quantum settings later in the notebook.

---

### Explanation

A Boolean function is a function that maps Boolean inputs to a Boolean output.
In this problem, each function takes **four Boolean inputs**, meaning there are *$2^4 = 16$* possible input combinations.
The function returns a single Boolean value (`True` or `False`) for each of these combinations.

The purpose of this problem is to randomly generate functions that are *guaranteed* to be either constant or balanced.
This mirrors the oracle model used in quantum computing, where the internal behaviour of the function is hidden and only its outputs can be observed through evaluation.

---

### Definitions

**Constant function**  
A Boolean function is constant if it returns the same output for every possible input.
For four input bits, this means the function returns either:
- `False` for all 16 input combinations, or
- `True` for all 16 input combinations.

**Balanced function**  
A Boolean function is balanced if it returns `True` for exactly half of all possible inputs and `False` for the other half.
With four input bits, this corresponds to:
- 8 inputs returning `True`
- 8 inputs returning `False`

---

### Simply Said

In simple terms, this problem creates a hidden Boolean function that:
- takes four `True`/`False` inputs,
- returns a single `True` or `False`,
- and is guaranteed to be either:
  - always the same (constant), or
  - evenly split between `True` and `False` (balanced).

The function type is chosen randomly, and the caller is not told which type was selected.

---

### Importance to Quantum Computing

This distinction between constant and balanced functions is fundamental to the Deutsch–Jozsa algorithm.
Classically, determining whether an unknown Boolean function is constant or balanced may require testing the function on many different inputs.

In contrast, the Deutsch–Jozsa algorithm can determine this global property with a **single evaluation** of the function when implemented as a quantum oracle. In simple terms, quantum computing cares more about the pattern than the inputs.  It can ask the question like *“Are the outputs the same, or do they cancel each other out?”* instead of looping through different inputs to evaluate the outputs.
Problem 1 establishes the classical foundation required to understand this quantum advantage by providing valid functions that satisfy the algorithm’s promise.

By constructing these functions explicitly, we will later compare classical and quantum approaches to solving the same problem.

---
### Example Behaviour

The table below shows the difference between constant and balanced Boolean functions using a small sample of input combinations.
Each input consists of four Boolean values, and the output is a single Boolean value.

| Input (x₁, x₂, x₃, x₄) | Constant Function Output | Balanced Function Output |
|------------------------|--------------------------|--------------------------|
| (False, False, False, False) | True  | False |
| (False, False, False, True)  | True  | True  |
| (False, False, True, False)  | True  | False |
| (False, True, False, False)  | True  | True  |
| (True, False, False, False)  | True  | False |
| (True, True, False, False)   | True  | True  |

In the constant case, the output is the same for every input.
In the balanced case, the outputs are split evenly between `True` and `False` across all possible inputs (only a subset is shown here).

---

### Alternative Approaches Considered

An alternative approach would have been to construct balanced functions using specific logical expressions (e.g., XOR combinations of inputs). While this can produce balanced behaviour, it does not automatically guarantee that exactly half of all 16 inputs return `True` without additional validation logic.

Another possibility would be to generate outputs randomly at each function call. However, this would violate the deterministic “oracle” assumption required by the Deutsch–Jozsa problem, where the function must have fixed behaviour.

For these reasons, a precomputed truth-table approach was selected. This guarantees correctness, and aligns cleanly with the mathematical definition of a Boolean function.

---

### References

- [IBM Quantum. *Deutsch–Jozsa Algorithm*.](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa)

- [Real Python. *Python Booleans*.](https://realpython.com/python-boolean/)

- [Python Software Foundation. *random : Generate pseudo-random numbers*.]( https://docs.python.org/3/library/random.html  )
 

- [Python Software Foundation. *itertools : Functions creating iterators for efficient looping*.]( https://docs.python.org/3/library/itertools.html)  


In [41]:
import random
import itertools

In [42]:
def random_constant_balanced():
    """
    Generate a random Boolean function of four Boolean inputs that is guaranteed
    to be either constant or balanced.

    The function takes four Boolean arguments and returns a Boolean.
    The function is randomly chosen to be either:
    
    - constant: always returns True or always returns False
    - balanced: returns True for exactly half of all possible inputs

    """

    # Generate all possible combinations of four Boolean inputs.
    # With four inputs, there are 2^4 = 16 possible combinations.
    inputs = list(itertools.product([False, True], repeat=4))

    # Randomly choose whether the function will be constant or balanced.
    function_type = random.choice(["constant", "balanced"])

    # Dictionary to store the output value for each input combination.
    # This acts as a classical truth table for the function.
    truth_table = {}

    if function_type == "constant":
        # Choose a single Boolean value to return for all inputs.
        value = random.choice([False, True])

        # Assign the same output value to every possible input.
        for inp in inputs:
            truth_table[inp] = value

    else:
        # For a balanced function, exactly half of the outputs must be True.
        # Shuffle the inputs to avoid any structural bias.
        random.shuffle(inputs)

        # Assign True to half of the inputs and False to the remainder.
        half = len(inputs) // 2
        for inp in inputs[:half]:
            truth_table[inp] = True
        for inp in inputs[half:]:
            truth_table[inp] = False

    # Define and return the Boolean function.
    # The function looks up its output from the precomputed truth table.
    def f(x1, x2, x3, x4):
        return truth_table[(x1, x2, x3, x4)]

    return f


In [43]:
# Testing
f = random_constant_balanced()

results = [f(*inp) for inp in itertools.product([False, True], repeat=4)]
print("True outputs:", results.count(True))
print("False outputs:", results.count(False))


True outputs: 8
False outputs: 8


## Problem 2: Classical Testing for Function Type

### Problem Context

**In Problem 1**, we generated Boolean functions that take four Boolean inputs and return a single Boolean output. Each function was guaranteed to be either **constant** (always returning the same value) or **balanced** (returning `True` for exactly half of the possible inputs).

This problem asks to determine - using a classical algorithm - whether a given function is constant or balanced.

At first glance, this may seem straightforward. However, this question is central to understanding why Deutsch's algorithm was historically significant. Before exploring the quantum solution, one must first understand the *classical cost* of solving the same problem.

The goal here is to understand how many times we must evaluate the function in order to be 100% certain of the answer.

>**What Is a Classical Algorithm?**
A classical algorithm is an algorithm that runs on a traditional (non-quantum) computer. It operates using classical bits, where information is represented as either 0 or 1.
Importantly for this exercise, Inputs are evaluated one at a time.

---

### Conceptual Explanation

To determine whether a function is constant or balanced using a classical computer, we must evaluate the function on specific input combinations and observe the outputs.

The function in this problem takes four Boolean inputs. Since each input can be either `True` or `False`, there are:

$$
2^4 = 16
$$

possible input combinations.

A classical strategy would proceed as follows:

1. Generate all possible input combinations.
2. Evaluate the function on each combination.
3. Record the outputs.
4. Analyze the pattern of the outputs.

If all outputs are identical, the function is **constant**.

If exactly half of the outputs are `True` and half are `False`, the function is **balanced**.

The key limitation of the classical approach is that it must gather information one input at a time. In the worst case, the algorithm may need to evaluate the function on every possible input before it can be completely certain of the result.

---

### Simply Said

We are given a function that takes four `True/False` inputs.  
We know in advance that it is either:

- Always the same (constant), or  
- Half `True` and half `False` (balanced).

To figure out which type it is using a classical computer, we must test the function on different input combinations and observe the outputs.

If every output matches, the function is constant.  
If the outputs are evenly split between `True` and `False`, the function is balanced.

The important question is:  
**How many times do we need to test the function before we can be completely certain?**

---

### Importance to Quantum Computing

---

### Example Behaviour

---

### Alternative Approaches Considered

---

### References
- [IBM Quantum. *Deutsch–Jozsa Algorithm*.](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa)
- [Deutsch-Josza (sic) problem implementation* : public Kaggle notebook with code and discussion of balanced vs constant function evaluation](https://www.kaggle.com/code/carlolepelaars/q4p-4-deutsch-josza?utm_source=chatgpt.com)
- []()

### The Code

In [44]:
def determine_constant_balanced(f):
    """
    Determine whether a Boolean function of four variables
    is constant or balanced.
    
    Returns:
        (result, calls) where
        result is "constant" or "balanced"
        calls is the number of times f was evaluated
    """
    
    inputs = itertools.product([False, True], repeat=4)
    
    # Evaluate the function on all possible inputs and count the outputs.
    # This will require 16 calls to the function, as there are 16 possible input combinations.
    outputs = []
    call_count = 0
    
    for inp in inputs:
        outputs.append(f(*inp))
        call_count += 1
    
    if all(o == outputs[0] for o in outputs):
        return "constant", f"It took {call_count} calls to determine the function is balanced."
    
    return "balanced", f"It took {call_count} calls to determine the function is balanced."

#### Testing Classical Approach Code

In [45]:
# Testing the determine_constant_balanced function with example functions

# Example constant function (always returns True)
def constant_example(a, b, c, d):
    return True


# Example balanced function (returns True if sum of inputs is even)
def balanced_example(a, b, c, d):
    return (a + b + c + d) % 2 == 0


# Test the determine_constant_balanced function
print("Constant test:", determine_constant_balanced(constant_example))
print("Balanced test:", determine_constant_balanced(balanced_example))

Constant test: ('constant', 'It took 16 calls to determine the function is balanced.')
Balanced test: ('balanced', 'It took 16 calls to determine the function is balanced.')


### Refining the Classical Approach

In the previous implementation, the function evaluated all 16 possible input combinations before returning a result. While this guarantees correctness, it does not reflect how a classical algorithm might behave in practice.

If a mismatch between outputs is detected early (for example, if the first call returns `True` and the second returns `False`), we can immediately conclude that the function is balanced. There is no need to continue evaluating the remaining inputs.

To better illustrate this idea, below is a refined version that stops early when sufficient information has been gathered. This allows us to observe both:

- The *best-case* number of function calls (when imbalance is detected quickly), and  
- The *worst-case* number of function calls (when the function is constant).

This refinement makes the classical cost of the problem more explicit.

### The Code

In [46]:
def determine_constant_balanced_early(f):
    inputs = itertools.product([False, True], repeat=4)
    
    first_output = None
    call_count = 0
    
    for inp in inputs:
        result = f(*inp)
        call_count += 1
        
        if first_output is None:
            first_output = result
        elif result != first_output:
            return "balanced", call_count
    
    return "constant", call_count

#### Testing Refined Classical Approach Code

In [47]:
# Testing the determine_constant_balanced_early function with example functions
# Example constant function
def constant_example(a, b, c, d):
    return True


# Balanced function (will likely trigger early stopping)
def balanced_example(a, b, c, d):
    return (a + b + c + d) % 2 == 0 # % 2 == 0 means it returns True for even sums and False for odd sums, which is balanced.


print("Constant test:", determine_constant_balanced_early(constant_example))
print("Balanced test:", determine_constant_balanced_early(balanced_example))

Constant test: ('constant', 16)
Balanced test: ('balanced', 2)
